In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
data_dir = Path("../data")
from src.annotation_process import process_bambi_annotations

In [ ]:
flights = list(data_dir.glob("*_matched_processed.mp4"))
print({f.name for f in flights})

In [ ]:
import cv2
import random
from pathlib import Path

def extract_detection_frames(
    video_path: Path, 
    yolo_dir: Path, 
    output_dir: Path, 
    bg_ratio: float = 0.10
):
    output_dir.mkdir(parents=True, exist_ok=True)
    flight_id = video_path.stem.replace("_matched_processed", "")

    annotated_frames = set()
    for txt_file in yolo_dir.glob("*.txt"):
        frame_idx = int(txt_file.stem.split("_")[-1])
        annotated_frames.add(frame_idx)

    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    all_frames = set(range(total_frames))
    empty_frames = list(all_frames - annotated_frames)
    sampled_bg = set(random.sample(empty_frames, int(len(empty_frames) * bg_ratio)))

    frames_to_extract = annotated_frames.union(sampled_bg)

    current_frame = 0
    saved_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if current_frame in frames_to_extract:
            gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)[:, 1024:]
            save_path = output_dir / f"{flight_id}_{current_frame:08d}.png"
            cv2.imwrite(str(save_path), gray_frame)
            saved_count += 1

            if current_frame in sampled_bg:
                (yolo_dir / f"{flight_id}_{current_frame:08d}.txt").touch()

        current_frame += 1

    cap.release()
    print(f"[{flight_id}] Extracted {len(annotated_frames)} animal frames and {len(sampled_bg)} background frames.")

In [ ]:
def extract_tracking_clips(video_path: Path, track_spans: dict, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    flight_id = video_path.stem.replace("_matched_processed", "")

    intervals = []
    for track_id, data in track_spans.items():
        start = max(0, data["first"] - 60) 
        end = data["last"] + 30
        intervals.append([start, end])

    intervals.sort(key=lambda x: x[0])

    merged_intervals = []
    for current in intervals:
        if not merged_intervals:
            merged_intervals.append(current)
        else:
            previous = merged_intervals[-1]
            if current[0] <= previous[1] + 60:
                previous[1] = max(previous[1], current[1])
            else:
                merged_intervals.append(current)

    cap = cv2.VideoCapture(str(video_path))
    current_frame = 0
    saved_count = 0
    interval_idx = 0

    while cap.isOpened() and interval_idx < len(merged_intervals):
        ret, frame = cap.read()
        if not ret:
            break

        active_start, active_end = merged_intervals[interval_idx]

        if active_start <= current_frame <= active_end:
            gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)[:, 1024:]
            save_path = output_dir / f"{flight_id}_{current_frame:08d}.png"
            cv2.imwrite(str(save_path), gray_frame)
            saved_count += 1

        if current_frame == active_end:
            interval_idx += 1

        current_frame += 1

    cap.release()
    print(f"[{flight_id}] Extracted {saved_count} total frames across {len(merged_intervals)} continuous merged clips.")

In [ ]:
import cv2
import random
from pathlib import Path

for video in flights:
    flight_id = video.name.replace("_matched_processed.mp4", "")

    ann_file = data_dir / f"{flight_id}_gt.txt"
    yolo_dir = data_dir / "labels" / flight_id
    yolo_frames_dir  = data_dir / "frames_detection" / flight_id / "thermal"
    track_frames_dir = data_dir / "frames_tracking"  / flight_id / "thermal"

    written, empty, track_spans = process_bambi_annotations(
        ann_file=ann_file,
        output_dir=yolo_dir,
        flight_id=flight_id,
        visibility_threshold=0.3,
        skip_propagated=True
    )
    print(f"\n--- Processing {flight_id} ---")
    print(f"Generated {written} YOLO labels. (Skipped {empty} empty frames).")

    print("Extracting YOLO detection frames...")
    extract_detection_frames(
        video_path=video,
        yolo_dir=yolo_dir,
        output_dir=yolo_frames_dir,
        bg_ratio=0.10
    )

    print("Extracting continuous tracking clips...")
    extract_tracking_clips(
        video_path=video,
        track_spans=track_spans,
        output_dir=track_frames_dir
    )

print("\nAll flights processed successfully!")